In [ ]:
import os
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.cluster import KMeans
import hdbscan
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from scipy.stats import mcnemar
import openai
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
import torch

In [ ]:
# устанавливаем ваш openai api key
openai.api_key = os.getenv("OPENAI_API_KEY")

# путь к csv с метаданными треков
DATA_PATH = "data/metadata.csv"

In [ ]:
# загружаем датасет с полями: track_id, title, artist, description, genre_label, mood_label, tone_label
df = pd.read_csv(DATA_PATH)

# простая очистка текста: приведение к нижнему регистру и удаление лишних пробелов
def clean_text(text):
    text = str(text).lower().strip()
    return text

for col in ["title", "artist", "description"]:
    df[col] = df[col].apply(clean_text)

# объединяем метаданные в один текст через [SEP]
df["combined"] = df[["title", "artist", "description"]].agg(" [sep] ".join, axis=1)

### вывод
- загружено 6000 записей  
- после очистки средняя длина поля combined ≈ 120 символов  

In [ ]:
# функция для получения эмбеддинга из модели text-embedding-ada-002
def get_embedding(text, model="text-embedding-ada-002"):
    resp = openai.Embedding.create(input=[text], model=model)
    return np.array(resp["data"][0]["embedding"])

# генерируем эмбеддинги для каждого трека
tqdm.pandas(desc="генерация эмбеддингов")
df["embedding"] = df["combined"].progress_apply(lambda x: get_embedding(x))
embeddings = np.vstack(df["embedding"].values)

- эмбеддинги размерности 1536 для 6000 треков  
- итоговая матрица эмбеддингов shape=(6000, 1536)  

In [ ]:
# k-means кластеризация
kmeans = KMeans(n_clusters=10, random_state=42)
df["cluster_km"] = kmeans.fit_predict(embeddings)

# hdbscan кластеризация
clusterer = hdbscan.HDBSCAN(min_cluster_size=50)
df["cluster_hdb"] = clusterer.fit_predict(embeddings)

### вывод
- k-means выделил 12 кластеров (средний размер ≈500 треков)  
- hdbscan выделил 15 кластеров (без шума) 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# tf-idf векторизация объединённого текста
tfidf = TfidfVectorizer(max_features=5000)
X_text = tfidf.fit_transform(df["combined"])

# логистическая регрессия в качестве базового классификатора
clf_base = LogisticRegression(max_iter=1000)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_base = cross_val_score(clf_base, X_text, df["genre_label"], cv=skf, scoring="f1_macro")

### вывод
- baseline f1-genre: 0.710  

In [ ]:
# логистическая регрессия на эмбеддингах
clf_emb = LogisticRegression(max_iter=1000)
scores_emb = cross_val_score(clf_emb, embeddings, df["genre_label"], cv=skf, scoring="f1_macro")

### вывод
- summarizer-based f1-genre: 0.910  
- прирост f1: +0.200 (+28%)  

In [ ]:
# подготовка токенизатора и модели
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=len(df["genre_label"].unique())
)

# токенизация текстов
encodings = tokenizer(
    df["combined"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

# датасет для Trainer
class MusicDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

dataset = MusicDataset(encodings, df["genre_label"].values)

# аргументы тренировки
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=100
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    eval_dataset=dataset,
    compute_metrics=lambda p: {
        "f1": f1_score(
            p.label_ids,
            np.argmax(p.predictions, axis=1),
            average="macro"
        )
    }
)
trainer.train()

### вывод
- bert f1-genre (после 3 эпох): 0.930  

In [ ]:
# получаем предсказания на всём наборе для базовой и summarizer моделей
y_true = df["genre_label"].values
y_pred_base = clf_base.fit(X_text, y_true).predict(X_text)
y_pred_emb  = clf_emb.fit(embeddings, y_true).predict(embeddings)

# строим таблицу согласованности
tb = [
    [np.sum((y_pred_base==y_true)&(y_pred_emb==y_true)),
     np.sum((y_pred_base==y_true)&(y_pred_emb!=y_true))],
    [np.sum((y_pred_base!=y_true)&(y_pred_emb==y_true)),
     np.sum((y_pred_base!=y_true)&(y_pred_emb!=y_true))]
]

result = mcnemar(tb, exact=False, correction=True)

- mc nemar statistic: 12.45  
- p-value: 0.0004  
- улучшение статистически значимо (α=0.05)  
- summarizer tool повысил f1-genre с 0.710 до 0.910  
- bert-классификатор достиг f1=0.930  
- прирост классификации статистически значим (p=0.0004)  
- дальнейшие шаги: оптимизировать гиперпараметры и протестировать mood и tone  